# Models evaluator notebook  

## Load and prepare data

### Load data

In [ ]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from rich.pretty import pprint
import matplotlib.pyplot as plt
from collections import defaultdict
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

IS_OPENAI_FILE = True # or False
IS_MAIN_DF = 'documents-02-08-sin05-conOpenAI.csv' # or False
OPENAI_PREDICTION_FILE = "predictions_openai-alldocsv1.pkl"
AYMURAI_JSONS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/' #'/Users/sofi/Desktop/collectiveAI/projects/AymurAI' # '/Users/sofi/Desktop/collectiveai/projects/AymurAI/'
DOCS_PATH = '/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' # '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/'

In [ ]:
if IS_OPENAI_FILE:
    try:
        with open(OPENAI_PREDICTION_FILE, "rb") as file:
            predictions = pickle.load(file)
    except Exception as e:
        print(f'There is no file named {OPENAI_PREDICTION_FILE}, please redefine OPENAI_PREDICTION_FILE variable')
else:
    print('There is no file of predictions')

if IS_MAIN_DF:
    df = pd.read_csv(IS_MAIN_DF)
else:
    print('Calculating main df from jsons from AimurAI outputs')
    json_files = [f for f in os.listdir(AYMURAI_JSONS_PATH) if f.endswith('.json')]
    para_files = [f for f in json_files if f.startswith('anonymization_paragraph')]
    para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in para_files], ignore_index=True)

    doc_files = [f for f in json_files if f.startswith('anonymization_document') and not 'paragraph' in f]
    doc_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_files], ignore_index=True)

    doc_para_files = [f for f in json_files if f.startswith('anonymization_document_paragraph')]
    doc_para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_para_files], ignore_index=True)

    doc_df = doc_df.copy()
    doc_para_df = doc_para_df.copy()
    doc_df['id'] = doc_df['id'].astype(str)
    doc_para_df['document_id'] = doc_para_df['document_id'].astype(str)


    merged_df = pd.merge(
        doc_para_df,
        doc_df[['id', 'created_at', 'name']],
        left_on='document_id',
        right_on='id',
        how='left'
    ).rename(columns={'id_x': 'id'}).drop(columns=['id_y'])

    df = pd.merge(
        para_df,
        merged_df,
        left_on='id',
        right_on='paragraph_id',
        how='left'
    )

In [ ]:
import numpy as np
np.sum(df['name'] != df['doc'])

### Function definitions

In [ ]:
def get_paragraph_predictions(row,predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= (row['start_char']) and (p['end_char'] <= row['end_char'])
    ]

def metrics_per_entity_by_span(df, is_openai = False):
    # acumula TP/FP/FN por entidad (label)
    counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for i in range(len(df)):
        # cada celda puede ser una lista de dicts; si falla, lista vacía
        try:
            val_list = eval(df.validation.iloc[i])
        except:
            val_list = []
        try:
            pred_list = eval(df.prediction.iloc[i])
        except:
            try:
                pred_list = df.prediction.iloc[i]
            except:
                pred_list = []
            
        # sets de spans por label: {(start, end), ...}
        val_spans = defaultdict(set)
        pred_spans = defaultdict(set)
        start_char = int(df.start_char.iloc[i])

        # VALIDATION (val)
        for v in (val_list or []):
            if not isinstance(v, dict):
                continue
            lab = (v.get('attrs', {}).get('aymurai_label', None)) or v.get('label', None)
            lab = lab.replace('CORREO_ELECTRÓNICO','CORREO_ELECTRONICO')
            s = v.get('attrs', {}).get('aymurai_alt_start_char', None)
            e = v.get('attrs', {}).get('aymurai_alt_end_char', None)
            if lab is not None and s is not None and e is not None:
                val_spans[lab].add((int(s), int(e)))
                #print('validation:')
                #print(lab, s, e)
        #print('val_spans: ', val_spans)

        # PREDICTIONS
        for p in (pred_list or []):
            if not isinstance(p, dict):
                continue
            lab = p.get('attrs', {}).get('aymurai_label', None) or p.get('label', None)
            lab = lab.replace('CORREO_ELECTRÓNICO','CORREO_ELECTRONICO')
            s = p.get('attrs', {}).get('aymurai_alt_start_char', None) or p.get('start_char', None)
            e = p.get('attrs', {}).get('aymurai_alt_end_char', None) or p.get('end_char', None)
            if lab is not None and s is not None and e is not None:
                if is_openai:
                    pred_spans[lab].add((int(s)-start_char, int(e)- start_char))
                else:
                    pred_spans[lab].add((int(s), int(e)))

                #print('prediction:')
                #print(lab, s, e)
        #print('pred_spans: ', pred_spans)
        # actualizar TP/FP/FN por label en esta fila
        for lab in set(list(val_spans.keys()) + list(pred_spans.keys())):
            vs = val_spans.get(lab, set())
            ps = pred_spans.get(lab, set())
            tp = len(ps & vs)
            fp = len(ps - vs)
            fn = len(vs - ps)
            counts[lab]['tp'] += tp
            counts[lab]['fp'] += fp
            counts[lab]['fn'] += fn

    # calcular métricas por label + micro/macro
    out = {'per_label': {}, 'micro': {}, 'macro': {}}
    micro_tp = micro_fp = micro_fn = 0

    for lab, c in counts.items():
        tp, fp, fn = c['tp'], c['fp'], c['fn']
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        support   = tp + fn  # cantidad de entidades de referencia (validation) para ese label
        out['per_label'][lab] = {'precision': precision, 'recall': recall, 'f1': f1, 'support': support}
        micro_tp += tp; micro_fp += fp; micro_fn += fn

    # micro
    micro_p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) > 0 else 0.0
    micro_r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) > 0 else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0
    out['micro'] = {'precision': micro_p, 'recall': micro_r, 'f1': micro_f, 'support': micro_tp + micro_fn}

    # macro (promedio simple entre labels presentes)
    if out['per_label']:
        L = len(out['per_label'])
        macro_p = sum(m['precision'] for m in out['per_label'].values()) / L
        macro_r = sum(m['recall']    for m in out['per_label'].values()) / L
        macro_f = sum(m['f1']        for m in out['per_label'].values()) / L
    else:
        macro_p = macro_r = macro_f = 0.0
    out['macro'] = {'precision': macro_p, 'recall': macro_r, 'f1': macro_f}

    return out
def metrics_for_label(df, target = 'text'):
    tp, fp, fn = 0, 0, 0
    for i in range(len(df)):
        try:
            val = eval(df.validation.iloc[i])[0]
        except:
            val = []
        try:
            pred = eval(df.prediction.iloc[i])[0]
        except:
            pred = []
        
        if target == 'text':
            text_val = val.get('attrs',None).get('aymurai_alt_text',None) if val else None
            text_pred= pred.get('text',None) if pred else None
            pred_set = set((text_pred,))
            val_set = set((text_val,))

        elif target == 'label':
            label_pred= pred.get('label',None) if pred else None
            label_val= val.get('attrs',None).get('aymurai_label',None) if val else None
            pred_set = set((label_pred,))
            val_set = set((label_val,))

        elif target == 'both':
            text_val = val.get('attrs',None).get('aymurai_alt_text',None) if val else None
            text_pred= pred.get('text',None) if pred else None
            label_pred= pred.get('label',None) if pred else None
            label_val= val.get('attrs',None).get('aymurai_label',None) if val else None
            pred_set = set((text_pred, label_pred))
            val_set = set((text_val, label_val))

        else:
            raise ValueError("Target must be one of 'text', 'label', or 'both'")
        if len(pred_set)>0 or len(val_set)>0: 
            tp += len(pred_set & val_set)
            fp += len(pred_set - val_set)
            fn += len(val_set - pred_set)

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / ( tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (recall + precision) > 0 else 0

    return {'recall': recall, 'precision': precision, 'f1': f1}

def _metrics_to_df(out):
    rows = []
    for lab, m in out.get('per_label', {}).items():
        rows.append({
            'label': str(lab),
            'precision': float(m.get('precision', 0) or 0),
            'recall':    float(m.get('recall', 0) or 0),
            'f1':        float(m.get('f1', 0) or 0),
            'support':   m.get('support', 0)
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        # Evita crash si support viene como NaN/None/str
        df['support'] = pd.to_numeric(df['support'], errors='coerce').fillna(0).astype(int)
        df = df.sort_values('support', ascending=False).reset_index(drop=True)
    return df

def plot_per_label_bars(out, top=None, sort_by='support', aclaration_title='', alphabetical=False):
    df = _metrics_to_df(out)
    if df.empty:
        print("No per-label data to plot."); 
        return
    if alphabetical:
        df = df.sort_values('label', ascending=True)
    elif sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=False)
    if top:
        df = df.head(top)

    labels = df['label'].tolist()
    x = np.arange(len(labels))
    width = 0.25

    fig, ax = plt.subplots(figsize=(max(6, len(labels)*0.9), 5),dpi = 150)
    ax.bar(x - width, df['precision'].values, width, label='Precision')
    ax.bar(x,         df['recall'].values,    width, label='Recall')
    ax.bar(x + width, df['f1'].values,        width, label='F1')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Score')
    ax.set_title('Per-label Precision / Recall / F1 '+ aclaration_title)
    ax.legend(loc = 'best')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_ylim(0, 1.4)

    # Anotar support arriba del rango actual del eje (sin asumir [0,1])
    y_top = ax.get_ylim()[1]
    y_text = y_top * 0.8
    for i, s in enumerate(df['support'].values):
        ax.text(i, y_text, f'n={int(s)}', ha='center', va='top', fontsize=11)

    plt.tight_layout()
    plt.show()

def plot_pr_scatter(out, min_support=1, annotate_top=None):
    df = _metrics_to_df(out)
    if df.empty:
        print("No per-label data to plot."); 
        return
    df = df[df['support'] >= min_support].copy()
    if df.empty:
        print("No points after min_support filter."); 
        return

    # tamaños relativos según support (evita división por cero)
    max_sup = df['support'].max()
    if max_sup <= 0:
        size = np.full(len(df), 100.0)
    else:
        size = 100 * (0.3 + 0.7 * (df['support'] / max_sup))

    fig, ax = plt.subplots(figsize=(6, 6),dpi = 150)
    ax.scatter(df['recall'].values, df['precision'].values, s=size, alpha=0.8)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_title('Precision vs Recall by Label (size ~ support)')
    ax.set_ylim(0, 1.4)

    # Anotar los top por support
    if annotate_top:
        top_df = df.sort_values('support', ascending=False).head(annotate_top)
    else:
        top_df = df.sort_values('support', ascending=False)

    for _, r in top_df.iterrows():
        ax.annotate(r['label'], (r['recall'], r['precision']), textcoords='offset points', xytext=(5, 5))

    plt.tight_layout()
    plt.show()

def plot_micro_macro(out):
    mm = out.get('micro', {}) or {}
    ma = out.get('macro', {}) or {}
    metrics = ['precision', 'recall', 'f1']
    micro_vals = [float(mm.get(m, 0) or 0) for m in metrics]
    macro_vals = [float(ma.get(m, 0) or 0) for m in metrics]

    x = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(6, 4),dpi = 150)
    b1 = ax.bar(x - width/2, micro_vals, width, label='Micro')
    b2 = ax.bar(x + width/2, macro_vals, width, label='Macro')

    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_title('Micro vs Macro')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_ylim(0, 1.4)

    # Anotar valores respetando límites del eje
    y_top = ax.get_ylim()[1]
    for i, v in enumerate(micro_vals):
        ax.text(i - width/2, min(v + 0.02*y_top, y_top*0.98), f'{v:.2f}', ha='center', va='bottom', fontsize=9)
    for i, v in enumerate(macro_vals):
        ax.text(i + width/2, min(v + 0.02*y_top, y_top*0.98), f'{v:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()



### Load joined_paragraphs from docs

In [ ]:
def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs


docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    print(d)
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars
    print(start_end_chars)
    print('\n')

In [ ]:
# TN es importante pero no cuenta en precision/recall/f1

# Example to visualize the actions of previuos functions

vs = {(312, 323), (400, 412), (55, 66), (238, 250)}
ps = {(312, 323), (423, 412), (55, 66), (238, 250)}
print(vs & ps, vs - ps, ps - vs)

## Performance metrics calculation

### Prepare dataset

Definition of files to analize: 
- df_main: AymurAI output documents 01 to 08 without 05, 

- df_main_dropdup: df_main without duplicate paragraphs,

- df_main_openai: df_main with the prediction of openai though Langextraxt (only if OPENAI_PREDICTION_FILE = True )

In [ ]:
#df_main = df[df['name'].fillna('').str.startswith('document') & (df['name']!= 'document-09.docx')]
#df_main.to_csv('documents-02-08-sin05.csv', index=False)
df_dropdup = pd.read_csv('documents-02-08-sin05.csv').drop_duplicates(subset='text', keep='first')

### Calculate metrics

#### With duplicated paragraph 

In [ ]:
out_ner = metrics_per_entity_by_span(df)
plot_per_label_bars(out_ner, top=None, sort_by='support', aclaration_title = 'with duplicated paragraphs',alphabetical = True)
plot_pr_scatter(out_ner, min_support=1, annotate_top=None)
plot_micro_macro(out_ner)

In [ ]:
set(df.paragraph_id) - set(df_dropdup.paragraph_id)

#### Without duplicated paragraph 

In [ ]:
df_dropdup = df.drop_duplicates(subset='text_x', keep='first')
out = metrics_per_entity_by_span(df_dropdup)
plot_per_label_bars(out, top=None, sort_by='support', aclaration_title = 'without duplicated paragraphs',alphabetical = True)
plot_pr_scatter(out, min_support=1, annotate_top=None)
plot_micro_macro(out)

In [ ]:
out['per_label'].keys()

#### OpenAI + LangExtract metrics

##### Set docs from pedictions_openai

In [ ]:
set(df_openai.prediction)

In [ ]:
predictions

In [ ]:
df_openai = df.copy().rename(columns={'prediction':'NER_prediction'})

#for d in set(df.name):
#d = 'document-04.docx'
#preds = predictions[d]
#df_d = df_openai[df_openai['name'] == d].reset_index(drop=True)
#df_d['prediction'] = df_d.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)

def get_paragraph_predictions(row, predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= row['start_char'] and p['end_char'] <= row['end_char']
    ]

df_openai['prediction'] = df_openai.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)


#errores = df_d[(df_d['validation'] != '[]') & (df_d['prediction'].apply(len) == 0)]

main_df = df_openai[df_openai.name == df_openai.doc]

out_openai = metrics_per_entity_by_span(main_df,is_openai=True)
plot_per_label_bars(out_openai, sort_by='support',alphabetical = True)
plot_pr_scatter(out_openai, min_support=1, annotate_top=18)
plot_micro_macro(out_openai)

In [ ]:
#df_openai.to_csv('df_openai.csv',index=False)

In [ ]:
#lambda x:  len([x.get('attrs', {}).get('aymurai_alt_start_char', None)for x in  eval(df_openai.validation.iloc[6]) ]) != len(set([x.get('attrs', {}).get('aymurai_alt_start_char', None)for x in  eval(df_openai.validation.iloc[6]) ]))

In [ ]:
check = lambda doc:  len([x.get('attrs', {}).get('aymurai_alt_start_char', None)for x in  eval(doc) ]) != len(set([x.get('attrs', {}).get('aymurai_alt_start_char', None)for x in  eval(doc) ]))

In [ ]:
df_openai.loc[df_openai.validation.map(lambda x: check(x))].shape

In [ ]:
main_df = df_openai[df_openai.name == df_openai.doc]

In [ ]:
main_df.loc[main_df.duplicated(subset=['paragraph_id','name'])].sort_values(['paragraph_id','name'])

In [ ]:
df_openai.loc[df_openai.duplicated(subset=['paragraph_id','name'])].sort_values(['paragraph_id','name'])

In [ ]:
df_openai= pd.read_csv('df_openai.csv')
df_openai

In [ ]:
df_openai.columns

In [ ]:
def metrics_per_entity_by_span(df, is_openai = False):
    # acumula TP/FP/FN por entidad (label)
    counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for i in range(len(df)):
        # cada celda puede ser una lista de dicts; si falla, lista vacía
        try:
            val_list = eval(df.validation.iloc[i])
        except:
            val_list = []
        try:
            pred_list = eval(df.prediction.iloc[i])
        except:
            try:
                pred_list = df.prediction.iloc[i]
            except:
                pred_list = []
            
        # sets de spans por label: {(start, end), ...}
        val_spans = defaultdict(set)
        pred_spans = defaultdict(set)
        start_char = int(df.start_char.iloc[i])

        # VALIDATION (val)
        for v in (val_list or []):
            if not isinstance(v, dict):
                continue
            lab = (v.get('attrs', {}).get('aymurai_label', None)) or v.get('label', None)
            lab = lab.replace('CORREO_ELECTRÓNICO','CORREO_ELECTRONICO')
            s = v.get('attrs', {}).get('aymurai_alt_start_char', None)
            e = v.get('attrs', {}).get('aymurai_alt_end_char', None)
            if lab is not None and s is not None and e is not None:
                val_spans[lab].add((int(s), int(e)))
                #print('validation:')
                #print(lab, s, e)
        #print('val_spans: ', val_spans)

        # PREDICTIONS
        for p in (pred_list or []):
            if not isinstance(p, dict):
                continue
            lab = p.get('attrs', {}).get('aymurai_label', None) or p.get('label', None)
            lab = lab.replace('CORREO_ELECTRÓNICO','CORREO_ELECTRONICO')
            s = p.get('attrs', {}).get('aymurai_alt_start_char', None) or p.get('start_char', None)
            e = p.get('attrs', {}).get('aymurai_alt_end_char', None) or p.get('end_char', None)
            if lab is not None and s is not None and e is not None:
                if is_openai:
                    pred_spans[lab].add((int(s)-start_char, int(e)- start_char))
                else:
                    pred_spans[lab].add((int(s), int(e)))

                #print('prediction:')
                #print(lab, s, e)
        #print('pred_spans: ', pred_spans)
        # actualizar TP/FP/FN por label en esta fila
        for lab in set(list(val_spans.keys()) + list(pred_spans.keys())):
            vs = val_spans.get(lab, set())
            ps = pred_spans.get(lab, set())
            tp = len(ps & vs)
            fp = len(ps - vs)
            fn = len(vs - ps)
            counts[lab]['tp'] += tp
            counts[lab]['fp'] += fp
            counts[lab]['fn'] += fn

    # calcular métricas por label + micro/macro
    out = {'per_label': {}, 'micro': {}, 'macro': {}}
    micro_tp = micro_fp = micro_fn = 0

    for lab, c in counts.items():
        tp, fp, fn = c['tp'], c['fp'], c['fn']
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        support   = tp + fn  # cantidad de entidades de referencia (validation) para ese label
        out['per_label'][lab] = {'precision': precision, 'recall': recall, 'f1': f1, 'support': support}
        micro_tp += tp; micro_fp += fp; micro_fn += fn

    # micro
    micro_p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) > 0 else 0.0
    micro_r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) > 0 else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0
    out['micro'] = {'precision': micro_p, 'recall': micro_r, 'f1': micro_f, 'support': micro_tp + micro_fn}

    # macro (promedio simple entre labels presentes)
    if out['per_label']:
        L = len(out['per_label'])
        macro_p = sum(m['precision'] for m in out['per_label'].values()) / L
        macro_r = sum(m['recall']    for m in out['per_label'].values()) / L
        macro_f = sum(m['f1']        for m in out['per_label'].values()) / L
    else:
        macro_p = macro_r = macro_f = 0.0
    out['macro'] = {'precision': macro_p, 'recall': macro_r, 'f1': macro_f}

    return out


### Compare models

In [ ]:
len(doc_paragraphs['document-04.docx'])

# Chequear parrafos

In [ ]:
[{doc:len(pars)} for doc, pars in doc_paragraphs.items()]

In [ ]:
main_df.groupby('name').count().paragraph_id

In [ ]:
import numpy as np
def compare_models(out_openai, out_ner, metric = 'F1',sort_by="support", top=None):
    df_o = _metrics_to_df(out_openai).set_index("label")
    df_n = _metrics_to_df(out_ner).set_index("label")
    
    # mergeando
    df = df_o.join(df_n, lsuffix="_openai", rsuffix="_ner", how="outer").fillna(0)

    # eliminar labels con soporte 0 en ambos
    df = df[(df["support_openai"] > 0) | (df["support_ner"] > 0)]

    # ordenar
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=False)
    if top:
        df = df.head(top)

    labels = df.index.tolist()
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(max(6,len(labels)*0.9),5),dpi=150)
    plt.grid(alpha = 0.5, linestyle = 'dashed')  
    if metric == 'F1': 
        bars1 = ax.bar(x - width/2, df["f1_openai"], width, color="darkgreen",label="OpenAI+LangExtract F1",edgecolor='black',linewidth = 1)
        bars2 = ax.bar(x + width/2, df["f1_ner"], width,color = "goldenrod" , label="NER F1",edgecolor='black',linewidth = 1)
        ax.set_title("Comparación F1 por entidad")
        ax.set_ylabel("F1")

    elif metric == 'recall':
        bars1 = ax.bar(x - width/2, df["recall_openai"], width, color="darkgreen",label="OpenAI+LangExtract recall",edgecolor='black',linewidth = 1)
        bars2 = ax.bar(x + width/2, df["recall_ner"], width,color = "goldenrod" , label="NER recall",edgecolor='black',linewidth = 1)
        ax.set_title("Comparación recall por entidad")
        ax.set_ylabel("recall")

    elif metric == 'precision':        
        bars1 = ax.bar(x - width/2, df["precision_openai"], width, color="darkgreen",label="OpenAI+LangExtract precision",edgecolor='black',linewidth = 1)
        bars2 = ax.bar(x + width/2, df["precision_ner"], width,color = "goldenrod" , label="NER precision",edgecolor='black',linewidth = 1)
        ax.set_title("Comparación precision por entidad")
        ax.set_ylabel("precision")

    # anotar supports
    for i, (b1, b2) in enumerate(zip(bars1, bars2)):
        s1 = int(df.iloc[i]["support_openai"])
        s2 = int(df.iloc[i]["support_ner"])
        h = np.max([b1.get_height(),b2.get_height()])
        ax.text(b1.get_x() + b1.get_width(), h+0.02, f"n={s1}", 
                ha="center", va="bottom", fontsize=14, color="black")

        #ax.text(b2.get_x() + b2.get_width()/2, h+0.02, f"n={s2}", 
        #        ha="center", va="bottom", fontsize=14, color="grey", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.legend()
    ax.set_ylim(0, 1.2)
    plt.tight_layout()
    plt.show()
    
    return df.reset_index()



In [ ]:
compare_models(out_openai, out_ner,metric = 'F1', sort_by="support_ner")

In [ ]:
compare_models(out_openai, out_ner,metric = 'recall',sort_by="support_ner")

In [ ]:
compare_models(out_openai, out_ner,metric = 'precision',sort_by="support_ner")

In [ ]:
from ast import literal_eval

def print_mismatches_both_models(df, labels=("DNI","BANCO","CORREO_ELECTRONICO","PER"), n_per_label=10):
    def to_list(cell, needs_eval=False):
        if cell is None or cell == "": return []
        if isinstance(cell, list): return [x for x in cell if isinstance(x, dict)]
        if isinstance(cell, dict): return [cell]
        # strings (CSV): intentar parsear
        try:
            parsed = literal_eval(str(cell))
            if isinstance(parsed, list):  return [x for x in parsed if isinstance(x, dict)]
            if isinstance(parsed, dict):  return [parsed]
        except Exception:
            return []
        return []

    def norm(items):
        out = []
        for it in items:
            if not isinstance(it, dict): 
                continue
            attrs = it.get("attrs") or {}
            lab = attrs.get("aymurai_label") or it.get("label")
            if lab == "CORREO_ELECTRÓNICO":  # normalizo acento
                lab = "CORREO_ELECTRONICO"
            txt = attrs.get("aymurai_alt_text") or it.get("extraction_text") or it.get("text") or ""
            if lab: out.append((str(lab), str(txt)))
        return out

    for label in labels:
        print(f"\n=== LABEL: {label} ===")
        shown_openai = 0
        shown_ner    = 0
        for _, row in df.iterrows():
            # CSV: validation y NER_prediction vienen como strings -> eval
            val_items  = to_list(row.get("validation"), needs_eval=True)
            ner_items  = to_list(row.get("NER_prediction"), needs_eval=True)
            # En tu CSV, prediction también viene como string; si en tu notebook ya es list/dict, igual funciona
            openai_items = to_list(row.get("prediction"), needs_eval=True)

            val_spans   = norm(val_items)
            ner_spans   = norm(ner_items)
            openai_spans= norm(openai_items)

            val_texts    = sorted(set(t for lab2, t in val_spans    if lab2 == label))
            openai_texts = sorted(set(t for lab2, t in openai_spans if lab2 == label))
            ner_texts    = sorted(set(t for lab2, t in ner_spans    if lab2 == label))

            snippet = (row.get('text_x') or '')[:220].replace('\n',' ')
            pid     = row.get('paragraph_id'); doc = row.get('name')

            # mismatches OpenAI
            if shown_openai < n_per_label and val_texts != openai_texts:
                shown_openai += 1
                print(f"[OpenAI {shown_openai}] paragraph_id={pid} | doc={doc}")
                print(f"  snippet: {snippet}")
                print(f"  val:     {val_texts}")
                print(f"  openai:  {openai_texts}\n")

            # mismatches NER
            if shown_ner < n_per_label and val_texts != ner_texts:
                shown_ner += 1
                print(f"[NER {shown_ner}] paragraph_id={pid} | doc={doc}")
                print(f"  snippet: {snippet}")
                print(f"  val:     {val_texts}")
                print(f"  ner:     {ner_texts}\n")

            if shown_openai >= n_per_label and shown_ner >= n_per_label:
                break


In [ ]:
print_mismatches_both_models(df_openai, labels=("DNI","BANCO","CORREO_ELECTRONICO","PER"), n_per_label=5)


In [ ]:
def sample_errors(df, label, n=3):
    samples = []
    for i,row in df.iterrows():
        val = eval(row["validation"]) if row["validation"] else []
        pred = row["prediction"] if row["prediction"] else []
        val_labels = [v.get("label") for v in val]
        pred_labels = [p.get("label") for p in pred]
        
        if label in val_labels and label not in pred_labels:
            samples.append(("FN", row["text_y"]))
        elif label not in val_labels and label in pred_labels:
            samples.append(("FP", row["text_y"]))
        elif label in val_labels and label in pred_labels:
            samples.append(("TP", row["text_y"]))
        
        if len(samples) >= n: break
    return samples


In [ ]:
sample_errors(df_openai,'PER')

In [ ]:
import random

def sample_cases(df, label, model="prediction", n=3):
    cases = {"TP": [], "FP": [], "FN": []}
    for _, row in df.iterrows():
        val = eval(row["validation"]) if row["validation"] else []
        pred = row[model] if row[model] else []
        
        val_spans = [v.get("label") for v in val if isinstance(v, dict)]
        pred_spans = [p.get("label") for p in pred if isinstance(p, dict)]
        
        text = row["text_x"][:300]  # recorte de párrafo para visualizar
        
        if label in val_spans and label not in pred_spans:
            cases["FN"].append(text)
        elif label not in val_spans and label in pred_spans:
            cases["FP"].append(text)
        elif label in val_spans and label in pred_spans:
            cases["TP"].append(text)
    
    # devolver muestra aleatoria
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}
sample_cases(df_openai,'PER')

import random

def sample_cases(df, label, model="prediction", n=3):
    # nombre dinámico para el campo de predicción principal
    pred_key = "openai" if model == "prediction" else ("ner" if model == "NER_prediction" else "predmodel")

    def _norm_label_text(item):
        if not isinstance(item, dict):
            return None, ""
        attrs = item.get("attrs", {}) or {}
        lab   = attrs.get("aymurai_label") or item.get("label")
        txt   = attrs.get("aymurai_alt_text") or item.get("extraction_text") or item.get("text", "")
        return lab, txt

    cases = {"TP": [], "FP": [], "FN": []}

    for _, row in df.iterrows():
        # validation
        val_raw = eval(row["validation"]) if row["validation"] else []
        val_spans = [_norm_label_text(v) for v in val_raw if isinstance(v, dict)]
        val_spans = [(lab, txt) for lab, txt in val_spans if lab is not None]

        # pred principal (según 'model')
        if model == "NER_prediction":
            pred_raw = eval(row[model]) if row.get(model) else []
        else:
            pred_raw = row.get(model) if row.get(model) else []
        pred_spans = [_norm_label_text(p) for p in pred_raw if isinstance(p, dict)]
        pred_spans = [(lab, txt) for lab, txt in pred_spans if lab is not None]

        # pred de ambos modelos (si existen las columnas)
        openai_raw = row.get("prediction")
        ner_raw    = row.get("NER_prediction")
        openai_list = openai_raw if (openai_raw and not isinstance(openai_raw, str)) else (eval(openai_raw) if openai_raw else [])
        ner_list    = ner_raw if (ner_raw and not isinstance(ner_raw, str)) else (eval(ner_raw) if ner_raw else [])

        openai_spans = [_norm_label_text(p) for p in (openai_list or []) if isinstance(p, dict)]
        openai_spans = [(lab, txt) for lab, txt in openai_spans if lab is not None]
        ner_spans    = [_norm_label_text(p) for p in (ner_list or []) if isinstance(p, dict)]
        ner_spans    = [(lab, txt) for lab, txt in ner_spans if lab is not None]

        # listas planas por label
        val_labels  = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]

        # snippet e info extra
        text     = (row.get("text_x") or "")#[:300]
        para_id  = row.get("paragraph_id", None)
        doc_name = row.get("name", None)

        # armar registro con val + ambas preds + clave dinámica
        base_rec = {
            "paragraph_id": para_id,
            "document": doc_name,
            "parrafo": text,
            "val":   [t for lab, t in val_spans if lab == label],
            "openai": [t for lab, t in openai_spans if lab == label],
            "ner":    [t for lab, t in ner_spans    if lab == label],
        }
        # setear pred principal bajo su clave dinámica
        base_rec[pred_key] = [t for lab, t in pred_spans if lab == label]

        # clasificar TP/FP/FN
        if (label in val_labels) and (label in pred_labels):
            cases["TP"].append(base_rec)
        elif (label in val_labels) and (label not in pred_labels):
            cases["FN"].append(base_rec)
        elif (label not in val_labels) and (label in pred_labels):
            cases["FP"].append(base_rec)

    # muestra aleatoria top-n por tipo
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}


In [ ]:
[pred for pred in predictions['document-08.docx'] if 'PINZON, HECTOR EZEQUIEL' in pred['text']]#=='PER']#

In [ ]:
predictions['document-02.docx'][1]['label']

In [ ]:
joined_texts['document-02.docx'][148:233]

In [ ]:
df_openai[df_openai['paragraph_id']=='c09fba61330852c4813fcbb2f3bec11e'].iloc[3].prediction

In [ ]:
df_openai[df_openai['paragraph_id']=='c09fba61330852c4813fcbb2f3bec11e']

In [ ]:
pprint(df_openai[df_openai['paragraph_id']=='c09fba61330852c4813fcbb2f3bec11e'].prediction.iloc[0])

In [ ]:

ejemplos = sample_cases(df_openai, "PER", model="prediction", n=5)
pprint(ejemplos)

In [ ]:
main_df[main_df['paragraph_id']=='334f8df8c4dc5512a11ffc5d6a2db9c9']

In [ ]:
[p for p in predictions['document-02.docx'] if p['start_char']>15139]

In [ ]:
ejemplos = sample_cases(df_openai, "BANCO", model="prediction", n=5)
pprint(ejemplos)

In [ ]:
ejemplos = sample_cases(main_df, "LOC", model="prediction", n=10)
pprint(ejemplos)

In [ ]:
ejemplos = sample_cases(df_openai, "PER", model="prediction", n=3)
pprint(ejemplos)

In [ ]:
ejemplos_per = sample_cases(df_openai, "PER", model="prediction", n=3)
print(ejemplos_per)

In [ ]:
df_openai[df_openai['paragraph_id']=='45b5531cdc7c5f7c9aaaca30e8b779a4']

In [ ]:
df_openai[df_openai['paragraph_id']=='134602b55c4754e3987718fa7907011b']

In [ ]:
ejemplos_tel = sample_cases(df_openai, "TELEFONO", model="prediction", n=3)
ejemplos_banco = sample_cases(df_openai, "BANCO", model="prediction", n=3)

In [ ]:
import random

def sample_cases(df, label, model="prediction", n=3):
    cases = {"TP": [], "FP": [], "FN": []}
    for _, row in df.iterrows():
        # convertir listas de dicts
        val = eval(row["validation"]) if row["validation"] else []
        if model == 'NER_prediction':
            pred = eval(row[model]) if row[model] else []
        else:
            pred = row[model] if row[model] else []
        
        # spans y textos de validación
        val_spans = [(v.get("label"), v.get("extraction_text") or v.get("text", "")) 
                     for v in val if isinstance(v, dict)]
        pred_spans = [(p.get("label"), p.get("extraction_text") or p.get("text", "")) 
                      for p in pred if isinstance(p, dict)]
        
        # listas planas de labels
        val_labels = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]
        
        # snippet de párrafo
        text = row["text_x"][:300]  
        
        # info extra
        para_id = row.get("paragraph_id", None)
        doc_name = row.get("name", None)
        
        # FN = estaba en val pero no en pred
        if label in val_labels and label not in pred_labels:
            cases["FN"].append({
                "paragraph_id": para_id,
                "document": doc_name,
                "parrafo": text,
                "gt": [t for lab, t in val_spans if lab == label],
                "pred": []
            })
        # FP = estaba en pred pero no en val
        elif label not in val_labels and label in pred_labels:
            cases["FP"].append({
                "paragraph_id": para_id,
                "document": doc_name,
                "parrafo": text,
                "gt": [],
                "pred": [t for lab, t in pred_spans if lab == label]
            })
        # TP = ambos lo tienen
        elif label in val_labels and label in pred_labels:
            cases["TP"].append({
                "paragraph_id": para_id,
                "document": doc_name,
                "parrafo": text,
                "gt": [t for lab, t in val_spans if lab == label],
                "pred": [t for lab, t in pred_spans if lab == label]
            })
    
    # tomar muestra aleatoria
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}



In [ ]:
def print_mismatches(df, label, model="prediction", n=10):
    from ast import literal_eval
    shown = 0

    for _, row in df.iterrows():
        # --- validation (requiere eval) ---
        val_cell = row.get("validation")
        if isinstance(val_cell, list):
            val_items = val_cell
        elif isinstance(val_cell, dict):
            val_items = [val_cell]
        else:
            try:
                val_items = literal_eval(str(val_cell)) if val_cell not in (None, "") else []
            except Exception:
                val_items = []

        # --- pred según modelo ---
        pred_cell = row.get(model)
        if model == "prediction":
            # ya es dict/list
            if isinstance(pred_cell, list):
                pred_items = pred_cell
            elif isinstance(pred_cell, dict):
                pred_items = [pred_cell]
            else:
                pred_items = []
        else:  # "NER_prediction" requiere eval
            if isinstance(pred_cell, list):
                pred_items = pred_cell
            elif isinstance(pred_cell, dict):
                pred_items = [pred_cell]
            else:
                try:
                    pred_items = literal_eval(str(pred_cell)) if pred_cell not in (None, "") else []
                except Exception:
                    pred_items = []

        # --- normalización mínima: (label, text) ---
        def norm(items):
            out = []
            for it in items:
                if not isinstance(it, dict):
                    continue
                attrs = it.get("attrs") or {}
                lab = attrs.get("aymurai_label") or it.get("label")
                txt = attrs.get("aymurai_alt_text") or it.get("extraction_text") or it.get("text") or ""
                if lab is not None:
                    out.append((str(lab), str(txt)))
            return out

        val_spans  = norm(val_items)
        pred_spans = norm(pred_items)

        val_texts  = sorted(set(t for lab, t in val_spans  if lab == label))
        pred_texts = sorted(set(t for lab, t in pred_spans if lab == label))

        if val_texts != pred_texts:
            shown += 1
            print(f"[{shown}] paragraph_id={row.get('paragraph_id')} | doc={row.get('name')}")
            snippet = (row.get('text_x') or '')[:220].replace('\n', ' ')
            print(f"  snippet: {snippet}")
            print(f"  val:      {val_texts}")
            print(f"  {model}:  {pred_texts}\n")
            if shown >= n:
                break


In [ ]:
print_mismatches(df_openai, label="BANCO", model="NER_prediction", n=5)  # NER


In [ ]:
df_openai[df_openai['paragraph_id']=='659ecd968d695ed19798ebd250c83dfe'].iloc[0].prediction

In [ ]:
print_mismatches(df_openai, label="TELEFONO", model="prediction", n=5)       # OpenAI


In [ ]:
ejemplos_tel = sample_cases(df_openai, "TELEFONO", model="prediction", n=3)
ejemplos_banco = sample_cases(df_openai, "BANCO", model="prediction", n=6)

from pprint import pprint
pprint(ejemplos_tel)


In [ ]:
pprint(ejemplos_banco)


In [ ]:
df_openai[df_openai['paragraph_id']=='659ecd968d695ed19798ebd250c83dfe'].prediction.iloc[0]

In [ ]:
ejemplos_correo = sample_cases(df_openai, "CORREO_ELECTRONICO", model="prediction", n=5)
pprint(ejemplos_correo)

In [ ]:
ejemplos_dni = sample_cases(df_openai, "DNI", model="prediction", n=5)
pprint(ejemplos_dni)

In [ ]:
eval(df_openai[df_openai['paragraph_id']=='fa3b25f7c3ec59cc8bbcb1ec16d72e18'].iloc[0].NER_prediction)

In [ ]:
eval(df_openai[df_openai['paragraph_id']=='fa3b25f7c3ec59cc8bbcb1ec16d72e18'].iloc[0].validation)

In [ ]:
df_openai[df_openai['paragraph_id']=='fa3b25f7c3ec59cc8bbcb1ec16d72e18'].iloc[0].prediction

In [ ]:
eval(df_openai[(df_openai['paragraph_id']=='de63b9abbe4c52ba877adf39d18b94d9')&(df_openai.name=='document-04.docx')].validation.iloc[1])

In [ ]:
ejemplos_dni = sample_cases(df_openai, "DNI", model="NER_prediction", n=5)
pprint(ejemplos_dni)

## Erros


In [ ]:
errores = df_openai[
    df_openai.apply(
        lambda row: len(eval(row['validation'])) if isinstance(row['validation'], str) else len(row['validation']),
        axis=1
    ) != df_openai['prediction'].apply(len)
].reset_index()


In [ ]:
#errores = df_openai[(df_openai['validation'] != '[]') & (df_openai['prediction'].apply(len) == 0)]

In [ ]:
df_openai[df_openai['paragraph_id']=='334f8df8c4dc5512a11ffc5d6a2db9c9']

In [ ]:
df_openai.iloc[8]

In [ ]:
df_openai[df_openai['paragraph_id']=='334f8df8c4dc5512a11ffc5d6a2db9c9'].validation.iloc[0]

In [ ]:
df_openai[df_openai['paragraph_id']=='334f8df8c4dc5512a11ffc5d6a2db9c9'].text_x.iloc[0][42:54]

In [ ]:
joined_texts['document-02.docx'][15139:15353]

In [ ]:
df_openai[df_openai['paragraph_id']=='334f8df8c4dc5512a11ffc5d6a2db9c9'].text_x.iloc[0]

In [ ]:
[pred['text'] for pred in predictions['document-02.docx'] if pred['start_char']>=15139 and pred['end_char']<= 15353] 

In [ ]:
eval(errores.iloc[i].validation)

## Old

Lo que haremos en esta sección es matcheat cada prediccion dentro del archivo de prediction, el cual contiene un unido conjunto de labels para todo le documento, con el parrafo al que corresponde. Para ello usaremos df (que tiene text, paragraph_id), y asociaremos primero un numero de orden y longitud a cada parrafo por documento. De esta manera podremos reescribir el output de langextract para poder comparar cada label con su respectivo validation value.

In [ ]:
df_main.head(2)

In [ ]:
df_main_openai = df_main.copy()

Create order and lenght paragraph columns

In [ ]:
df_main_openai['paragraph_order'] = df_main_openai.groupby('name').cumcount()
df_main_openai['paragraph_length'] = df_main_openai['text'].apply(len)
df_main_openai.head(3)

In [ ]:
# Chequeo que el orden de los parrafos este bien por documento
np.sum(df_main_openai.groupby(['paragraph_id','paragraph_order']).nunique().text != 1)

Creat start_char and end_char columns

In [ ]:
#df_main_openai = df_main_openai.sort_values(['name', 'paragraph_order'], ignore_index=True)
df_main_openai['end_char'] = df_main_openai.groupby('name')['paragraph_length'].cumsum()
df_main_openai['start_char'] = df_main_openai['end_char'] - df_main_openai['paragraph_length']
df_main_openai.head(5)

In [ ]:

errores = df_main_openai[(df_main_openai['validation'] != '[]') & (df_main_openai['prediction_openai_bis'].apply(len) == 0)]


In [ ]:
errores['validation_len'] = errores['validation'].apply(lambda x: len(eval(x)) if isinstance(x, str) else len(x))
errores['prediction_len'] = errores['prediction'].apply(lambda x: len(x))
errores_diff = errores[errores['validation_len'] != errores['prediction_len']]
errores_diff[['validation', 'prediction', 'validation_len', 'prediction_len']]

In [ ]:
errores = df_openai[(df_openai['validation'] != '[]') & (df_openai['prediction'].apply(len) == 0)]
errores

In [ ]:
for i in range(len(errores)):
    print('pred:',errores.iloc[i].prediction)
    print('val:',errores.iloc[i].validation)
    print('id:',errores.iloc[i].paragraph_id)
    print('doc:',errores.iloc[i].doc)

In [ ]:
target_text = df_openai.loc[df_openai['paragraph_id'] == '334f8df8c4dc5512a11ffc5d6a2db9c9', 'text_x'].iloc[0]
df_openai[df_openai['text_y'] == target_text]